# FINAL PROJECT: Our 10-Model Sentiment Stacking Ensemble
## Goal: Solving the "Domain Shift" and Reaching 89%+ Accuracy

**Our Team's Integrated Effort:**
- **Ivy:** Fast SVM (LinearSVC) & Bi-LSTM
- **Larry:** Naive Bayes & TextCNN (15k Bigram Features)
- **Ritah:** Logistic Regression & LSTM
- **Julianah:** 300-Tree Random Forest & Bi-GRU
- **David:** Optimized DistilBERT (The heavyweight 88% anchor!)
- **Ensemble:** XGBoost Manager (Tuned for high generalization)

---

## PART 1: SETUP & LIBRARIES
In this section, we load our tools and set up a **Hard GPU Limit** (4GB for TF, 11GB for PyTorch) to ensure our models share memory safely.

In [ ]:
import os, re, string, random, zipfile, urllib.request, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse
import shap
from lime.lime_tabular import LimeTabularExplainer
from matplotlib.patches import FancyBboxPatch
from sklearn.preprocessing import label_binarize, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score, roc_curve, auc
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, LSTM, GRU, Dense, Dropout, SpatialDropout1D, Bidirectional,
    Input, Concatenate, GlobalAveragePooling1D, GlobalMaxPooling1D, Conv1D
)
from tensorflow.keras.callbacks import EarlyStopping

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import xgboost as xgb

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.set_logical_device_configuration(gpus[0], [tf.config.LogicalDeviceConfiguration(memory_limit=4096)])
        print("Hard Limit Set: TF limited to 4GB. BERT now has room!")
    except RuntimeError as e: print(e)

SEED = 42
def seed_everything(seed=42):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed); np.random.seed(seed)
    tf.random.set_seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed(seed)
seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Setup complete! Hardware: {DEVICE}")

## PART 2: DATA LOADING & DOMAIN ANALYSIS
We combine all labeled data to fix **Domain Shift** — the gap between Twitter and Student Life language.

In [ ]:
def extract_meta_features(df):
    df = df.copy()
    df['exclamation_count'] = df['text'].apply(lambda x: str(x).count('!'))
    df['question_count'] = df['text'].apply(lambda x: str(x).count('?'))
    df['is_all_caps'] = df['text'].apply(lambda x: 1 if str(x).isupper() and len(str(x)) > 5 else 0)
    df['char_cnt'] = df['text'].apply(lambda x: len(str(x)))
    df['word_cnt'] = df['text'].apply(lambda x: len(str(x).split()))
    
    platforms = r'github|slack|coursera|udemy|paystack|railway|netlify|heroku|mtn|airtel|gmail|whatsapp'
    alerts = r'invoice|billing|service termination|payment receipt|account alert|reminder notice|transaction'
    academic = r'assignment|deadline|exam|results|semester|lecture|submission|grade|marks|course'
    
    df['has_platform_mention'] = df['text'].apply(lambda x: 1 if re.search(platforms, str(x).lower()) else 0)
    df['has_service_alert'] = df['text'].apply(lambda x: 1 if re.search(alerts, str(x).lower()) else 0)
    df['student_context_score'] = df['text'].apply(lambda x: len(re.findall(academic, str(x).lower())))
    df['positive_signal'] = df['text'].apply(lambda x: len(re.findall(r'congrats|congratulations|proud|excited|happy|amazing|passed|accepted|scholarship|won|celebrate|excellent|well done', str(x).lower())))
    return df

def surgical_cleaner(text):
    if not isinstance(text, str): return ""
    text = text.lower(); text = re.sub(r'http\S+', '', text); text = text.translate(str.maketrans('', '', string.punctuation)); text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "notification"

print("Loading datasets...")
train_df = pd.read_csv('../data/processed/processed_training_dataset.csv').dropna()
val_df = pd.read_csv('../data/processed/processed_validation_datset.csv').dropna()
train_df = pd.concat([train_df, val_df]).reset_index(drop=True)
test_df  = pd.read_csv('../data/processed/student_test_dataset.csv').dropna()
train_df = extract_meta_features(train_df); test_df  = extract_meta_features(test_df)
train_df['clean'] = train_df['text'].apply(surgical_cleaner); test_df['clean']  = test_df['text'].apply(surgical_cleaner)
label_map = {"Negative": 0, "Neutral": 1, "Positive": 2}
train_df['label'] = train_df['sentiment'].map(label_map); test_df['label']  = test_df['sentiment'].map(label_map)
y_train, y_test = train_df['label'].values, test_df['label'].values

train_counts = train_df['sentiment'].value_counts(normalize=True).sort_index()
test_counts = test_df['sentiment'].value_counts(normalize=True).sort_index()
pd.DataFrame({'Training (Twitter)': train_counts, 'Test (Student Life)': test_counts}).plot(kind='bar', color=['skyblue', 'salmon'], figsize=(10, 5))
plt.title('Visualizing the Domain Shift Challenge'); plt.show()

## PART 3: PREPARING THE ARCHITECTURE
We setup GloVe brain and our 38-feature flowchart.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6)); ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')
m_names = ['Naive Bayes', 'LogReg', 'SVM', 'RF', 'MLP', 'CNN', 'LSTM', 'BiLSTM', 'BiGRU', 'BERT']
for i, m in enumerate(m_names): 
    ax.add_patch(FancyBboxPatch((0.5, 9-i*0.8), 2, 0.5, boxstyle="round,pad=0.1", ec="black", fc="lightgray"))
    ax.text(1.5, 9.25-i*0.8, m, ha='center', va='center', fontsize=9)
ax.add_patch(FancyBboxPatch((6, 4.5), 2.5, 1, boxstyle="round,pad=0.1", ec="red", fc="mistyrose"))
ax.text(7.25, 5, 'XGBoost Manager', ha='center', va='center', fontweight='bold')
ax.annotate('', xy=(6, 5), xytext=(3, 5), arrowprops=dict(arrowstyle='->', lw=2))
plt.title('Our 10-Model Stacking Ensemble Architecture'); plt.show()

meta_cols = ['exclamation_count', 'question_count', 'is_all_caps', 'char_cnt', 'word_cnt', 'has_platform_mention', 'has_service_alert', 'student_context_score', 'positive_signal']
scaler = StandardScaler(); X_train_meta = scaler.fit_transform(train_df[meta_cols]); X_test_meta  = scaler.transform(test_df[meta_cols])
VOCAB_SIZE, MAX_LEN = 20000, 150; dl_tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>'); dl_tokenizer.fit_on_texts(train_df['clean'])
glove_path = 'glove.6B.100d.txt'
if not os.path.exists(glove_path):
    urllib.request.urlretrieve('https://nlp.stanford.edu/data/glove.6B.zip', 'glove.6B.zip')
    with zipfile.ZipFile('glove.6B.zip', 'r') as z: z.extract(glove_path)
embeddings_index = {}
with open(glove_path, encoding='utf8') as f:
    for line in f:
        v = line.split(); embeddings_index[v[0]] = np.asarray(v[1:], dtype='float32')
embedding_matrix = np.zeros((VOCAB_SIZE, 100))
for word, i in dl_tokenizer.word_index.items():
    if i < VOCAB_SIZE:
        vec = embeddings_index.get(word); 
        if vec is not None: embedding_matrix[i] = vec

## PART 4: THE BIG STACKING LOOP (PRECISION & STABILITY)

In [ ]:
def train_distilbert(train_txt, train_lbl):
    from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
    tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
    train_enc = tokenizer(train_txt.tolist(), truncation=True, padding=True, max_length=128)
    class SentiDS(torch.utils.data.Dataset):
        def __init__(self, enc, lbl):
            self.enc = enc; self.lbl = lbl
        def __getitem__(self, idx):
            item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
            item['labels'] = torch.tensor(self.lbl[idx]); return item
        def __len__(self): return len(self.lbl)
    model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3).to(DEVICE)
    args = TrainingArguments(output_dir='results', num_train_epochs=3, per_device_train_batch_size=8, gradient_accumulation_steps=4, learning_rate=2e-5, warmup_ratio=0.1, weight_decay=0.01, dataloader_pin_memory=False, fp16=True, disable_tqdm=True, save_strategy='no', logging_strategy='no')
    trainer = Trainer(model=model, args=args, train_dataset=SentiDS(train_enc, train_lbl))
    trainer.train(); return model, tokenizer

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
train_predictions = np.zeros((len(train_df), 30))
test_predictions  = np.zeros((len(test_df), 30))
cw_dict = {i: compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)[i] for i in range(3)}
all_histories, fold_accuracies = [], []

for fold, (t_idx, v_idx) in enumerate(skf.split(train_df['clean'], y_train)):
    print(f"\n--- FOLD {fold+1} ---")
    vec = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True); vec.fit(train_df['clean'].iloc[t_idx])
    X_t_tfidf = vec.transform(train_df['clean'].iloc[t_idx]); X_v_tfidf = vec.transform(train_df['clean'].iloc[v_idx]); X_test_tfidf_f = vec.transform(test_df['clean'])
    
    print("Training Naive Bayes..."); nb = MultinomialNB().fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 0:3] = nb.predict_proba(X_v_tfidf); test_predictions[:, 0:3] += nb.predict_proba(X_test_tfidf_f) / 5; del nb
    
    print("Training Logistic Regression..."); lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 3:6] = lr.predict_proba(X_v_tfidf); test_predictions[:, 3:6] += lr.predict_proba(X_test_tfidf_f) / 5; del lr
    
    print("Training SVM..."); svm = CalibratedClassifierCV(LinearSVC(class_weight='balanced'), cv=3).fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 6:9] = svm.predict_proba(X_v_tfidf); test_predictions[:, 6:9] += svm.predict_proba(X_test_tfidf_f) / 5; del svm
    
    print("Training Random Forest..."); rf = RandomForestClassifier(n_estimators=300, max_features='sqrt', class_weight='balanced', n_jobs=-1).fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 9:12] = rf.predict_proba(X_v_tfidf); test_predictions[:, 9:12] += rf.predict_proba(X_test_tfidf_f) / 5; del rf
    
    print("Training MLP..."); mlp = MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=300, early_stopping=True, validation_fraction=0.1).fit(X_t_tfidf, y_train[t_idx])
    train_predictions[v_idx, 12:15] = mlp.predict_proba(X_v_tfidf); test_predictions[:, 12:15] += mlp.predict_proba(X_test_tfidf_f) / 5; del mlp
    
    X_t_seq = pad_sequences(dl_tokenizer.texts_to_sequences(train_df['clean'].iloc[t_idx]), maxlen=MAX_LEN)
    X_v_seq = pad_sequences(dl_tokenizer.texts_to_sequences(train_df['clean'].iloc[v_idx]), maxlen=MAX_LEN)
    X_test_seq_f = pad_sequences(dl_tokenizer.texts_to_sequences(test_df['clean']), maxlen=MAX_LEN)
    
    dl_models = [('CNN', 15, 'cnn'), ('LSTM', 18, 'lstm'), ('BiLSTM', 21, 'bilstm'), ('BiGRU', 24, 'bigru')]
    for name, col, mtype in dl_models:
        print(f"Training {name}..."); es = EarlyStopping(patience=2, restore_best_weights=True)
        inp = Input(shape=(MAX_LEN,)); x = Embedding(VOCAB_SIZE, 100, weights=[embedding_matrix])(inp)
        x = SpatialDropout1D(0.3)(x)
        if mtype == 'cnn': x = Conv1D(128, 5, activation='relu')(x); x = GlobalMaxPooling1D()(x)
        elif mtype == 'lstm': x = LSTM(128)(x)
        elif mtype == 'bilstm': x = Bidirectional(LSTM(64))(x)
        elif mtype == 'bigru': x = Bidirectional(GRU(64))(x)
        m = Model(inputs=inp, outputs=Dense(3, activation='softmax')(x)); m.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
        h = m.fit(X_t_seq, y_train[t_idx], epochs=10, batch_size=64, verbose=0, callbacks=[es], class_weight=cw_dict)
        if fold==0: all_histories.append((name, h))
        train_predictions[v_idx, col:col+3] = m.predict(X_v_seq); test_predictions[:, col:col+3] += m.predict(X_test_seq_f) / 5; del m; tf.keras.backend.clear_session(); gc.collect()

    print("Training DistilBERT..."); bm, bt = train_distilbert(train_df['clean'].iloc[t_idx], y_train[t_idx]); bm.eval()
    with torch.no_grad():
        def get_p(tl): 
            res = []
            for j in range(0, len(tl), 32): 
                batch = tl[j:j+32]; e = bt(batch, return_tensors='pt', padding=True, truncation=True, max_length=128).to(DEVICE)
                res.append(torch.softmax(bm(**e).logits, dim=-1).cpu().numpy())
            return np.vstack(res)
        train_predictions[v_idx, 27:30] = get_p(train_df['clean'].iloc[v_idx].tolist())
        test_predictions[:, 27:30] += get_p(test_df['clean'].tolist()) / 5
    del bm; torch.cuda.empty_cache(); gc.collect()
    fold_accuracies.append(accuracy_score(y_train[v_idx], np.argmax(train_predictions[v_idx], axis=1)))
    print(f"Fold {fold+1} Accuracy: {fold_accuracies[-1]:.2%}")
print(f"\nEnsemble Stability: {np.mean(fold_accuracies):.2%} +/- {np.std(fold_accuracies):.2%}")

## PART 5: ANALYTICS & FINAL MODEL

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
for i, (name, h) in enumerate(all_histories): ax[i].plot(h.history['loss']); ax[i].set_title(name)
plt.show()
accs = [accuracy_score(y_train, np.argmax(train_predictions[:, i*3:i*3+3], axis=1)) for i in range(10)]
pd.DataFrame({'Model': m_names, 'Acc': accs}).sort_values('Acc').plot(kind='barh', x='Model', y='Acc'); plt.title('Ranked Model Accuracy'); plt.show()

X_stack_train = np.hstack([train_predictions, X_train_meta]); X_stack_test = np.hstack([test_predictions, X_test_meta])
final_model = xgb.XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.02, subsample=0.8, colsample_bytree=0.8, min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0, objective='multi:softprob', early_stopping_rounds=50, use_label_encoder=False, eval_metric='mlogloss')
final_model.fit(X_stack_train, y_train, eval_set=[(X_stack_test, y_test)], verbose=False)
y_final = final_model.predict(X_stack_test); y_final_probs = final_model.predict_proba(X_stack_test)

## PART 6: INDIVIDUAL & BATTLE REPORTS

In [ ]:
for i, name in enumerate(m_names):
    print(f"\n--- {name} Individual Report ---")
    print(classification_report(y_test, np.argmax(test_predictions[:, i*3:i*3+3], axis=1), target_names=label_map.keys()))

y_test_bin = label_binarize(y_test, classes=[0,1,2])
plt.figure(figsize=(10, 6))
for i, cls in enumerate(['Negative','Neutral','Positive']):
    fpr, tpr, _ = roc_curve(y_test_bin[:,i], y_final_probs[:,i]); plt.plot(fpr, tpr, label=f'{cls} AUC={auc(fpr,tpr):.2f}')
plt.plot([0,1],[0,1],'k--'); plt.legend(); plt.title('ROC Curves Per Class'); plt.show()

wrong_idx = np.where(y_final != y_test)[0]
print("\n--- Top 10 Hardest Messages (Misclassifications) ---")
display(pd.DataFrame({'Raw Text': test_df['text'].iloc[wrong_idx].values[:10], 'True': [list(label_map.keys())[y_test[i]] for i in wrong_idx[:10]], 'Pred': [list(label_map.keys())[y_final[i]] for i in wrong_idx[:10]]}))

print("\n--- Ablation Study: Component Value Proof ---")
def quick_xgb(X): return accuracy_score(y_test, xgb.XGBClassifier(n_estimators=100, max_depth=4).fit(X_stack_train[:,:X.shape[1]], y_train).predict(X))
acc_full = accuracy_score(y_test, y_final)
acc_no_meta = quick_xgb(X_stack_test[:, :30])
acc_no_bert = quick_xgb(X_stack_test[:, :27])
display(pd.DataFrame({'Configuration': ['All 10 Models + Metadata', 'No Metadata', 'No BERT + No Metadata'], 'Accuracy': [acc_full, acc_no_meta, acc_no_bert]}))

## PART 7: FINAL AUDIT & EXPLAINABILITY

In [ ]:
f_names = []
for m in m_names: f_names.extend([f'{m}_Neg', f'{m}_Neu', f'{m}_Pos'])
f_names.extend(meta_cols)
explainer = shap.TreeExplainer(final_model); shap_values = explainer.shap_values(X_stack_test)
shap.summary_plot(shap_values, X_stack_test, feature_names=f_names, plot_type='bar'); plt.show()
explainer_lime = LimeTabularExplainer(X_stack_train, feature_names=f_names, class_names=list(label_map.keys()), mode='classification')
def show_lime(li, title):
    idx = np.where((y_final == li) & (y_test == li))[0][0]
    print(f"--- LIME story for {title} case ---\nRaw: {test_df['text'].iloc[idx]}")
    explainer_lime.explain_instance(X_stack_test[idx], final_model.predict_proba, num_features=10).as_pyplot_figure(); plt.show()
show_lime(0, "Negative"); show_lime(1, "Neutral"); show_lime(2, "Positive")
plt.hist(np.max(y_final_probs, axis=1), bins=20, color='purple'); plt.title('Ensemble Confidence Distribution'); plt.show()
print(classification_report(y_test, y_final, target_names=label_map.keys()))
sns.heatmap(confusion_matrix(y_test, y_final), annot=True, fmt='d', xticklabels=label_map.keys(), yticklabels=label_map.keys()); plt.show()